In [ ]:
# In[1]:

import json

# Load the task.json file
task_file_path = 'rca100/cases/t094/task.json'
with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert context indicates that the alert was triggered by the operation `inventory::/api/v1/inventory/batch-reconciliation` in the APM domain. The alert time window is from `2026-04-04T22:37:09.273091+08:00` to `2026-04-04T22:40:48+08:00`. However, the alert title is missing or not provided.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T22:37:09.273091+08:00', 'end': '2026-04-04T22:40:48+08:00'}, {'entity_id': '2d5ef468202802748423656575955265', 'entity_name': 'inventory::/api/v1/inventory/batch-reconciliation', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t094/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Ensure 'time' and 'value' columns are numeric
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Calculate global thresholds (e.g., P95) for each KPI
global_thresholds = metrics_data.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
global_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

global_thresholds

```
Out[2]:
```
The global thresholds (P95) for each KPI have been successfully calculated for all possible root cause components (services and nodes). The results include metrics such as `apiserver_concurrent_connections`, `apiserver_requests_per_second`, `cluster_failed_node_count`, and service-specific metrics like `avg_request_latency_seconds` and `request_count` for operations such as `shipping::oteldemo.ShippingService/GetQuote`. The full table contains 300 rows, providing P95 thresholds for various metrics across different entities.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name                               metric  p95_threshold
0                                                     apiserver_concurrent_connections       5.000000
1                                                   apiserver_non_2xx_read_request_qps       0.088373
2                                                  apiserver_non_2xx_write_request_qps       0.087954
3                                                        apiserver_requests_per_second      26.256153
4                                                            cluster_failed_node_count       0.000000
..                                            ...                                  ...            ...
295                                      shipping                             workload   12398.750000
296   shipping::oteldemo.ShippingService/GetQuote          avg_request_latency_seconds       0.002897
297   shipping::oteldemo.ShippingService/GetQuote                        request_count    6201.550000
298  shipping::oteldemo.ShippingService/ShipOrder          avg_request_latency_seconds       0.000009
299  shipping::oteldemo.ShippingService/ShipOrder                        request_count    6195.850000

[300 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
start_time = datetime.fromisoformat('2026-04-04T22:37:09.273091+08:00').astimezone(timezone)
end_time = datetime.fromisoformat('2026-04-04T22:40:48+08:00').astimezone(timezone)

# Convert the time window to microseconds for filtering
start_time_us = int(start_time.timestamp() * 1_000_000)
end_time_us = int(end_time.timestamp() * 1_000_000)

# Filter metrics data within the time window
filtered_metrics = metrics_data[(metrics_data['time'] >= start_time_us) & (metrics_data['time'] <= end_time_us)]

# Merge filtered metrics with global thresholds to identify anomalies
merged_data = filtered_metrics.merge(global_thresholds, on=['entity_name', 'metric'], how='left')
anomalies = merged_data[merged_data['value'] > merged_data['p95_threshold']]

anomalies

```
Out[3]:
```
Anomalies were identified during the specified time window by comparing KPI values against their P95 thresholds. Examples include high `node_memory_usage_rate` and `node_disk_usage_rate` for Kubernetes nodes, as well as elevated `error_rate` for instances of services like `frontend` and `frontend-proxy`. A total of 508 anomalies were detected across various metrics and components.

The original code execution output of IPython Kernel is also provided below for reference:

time domain    entity_set                         entity_id                      entity_name                  metric      value                                     metric_set_id service  p95_threshold
38    1775313549000000    k8s      k8s.node                                                                     node_memory_usage_rate  91.363982  k8s@metric_set@k8s.metric.high_level_metric_node    None      91.341733
39    1775313579000000    k8s      k8s.node                                                                     node_memory_usage_rate  91.378649  k8s@metric_set@k8s.metric.high_level_metric_node    None      91.341733
40    1775313609000000    k8s      k8s.node                                                                     node_memory_usage_rate  91.481706  k8s@metric_set@k8s.metric.high_level_metric_node    None      91.341733
41    1775313639000000    k8s      k8s.node                                                                     node_memory_usage_rate  91.571014  k8s@metric_set@k8s.metric.high_level_metric_node    None      91.341733
392   1775313459000000    k8s      k8s.node                                                                       node_disk_usage_rate  34.929127  k8s@metric_set@k8s.metric.high_level_metric_node    None      34.842755
...                ...    ...           ...                               ...                              ...                     ...        ...                                               ...     ...            ...
4572  1775313639000000    apm  apm.instance  009ec66a350bc3e5210aea3dc6e6bdfd        frontend-6df7d557f5-sgfcb              error_rate   0.863445            apm@metric_set@apm.metric.apm.instance    None       0.857070
4574  1775313639000000    apm  apm.instance  7fb113e159d1fa7ccdfc641acf1870ad       inventory-794d9f8f85-hhjtj              error_rate   0.003663            apm@metric_set@apm.metric.apm.instance    None       0.003651
4579  1775313639000000    apm  apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd              error_rate   0.884289            apm@metric_set@apm.metric.apm.instance    None       0.870669
4584  1775313639000000    apm  apm.instance  82ae8fb5979d5759841affc9d937a0d4        frontend-6df7d557f5-nbvwh              error_rate   0.862228            apm@metric_set@apm.metric.apm.instance    None       0.858396
4599  1775313639000000    apm  apm.instance  fa548039d9a7b76db9ec70507e734faf  frontend-proxy-67d64dc4f8-8c99b              error_rate   0.857933            apm@metric_set@apm.metric.apm.instance    None       0.848196

[508 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t094/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime' and 'endTime' columns are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Convert the time window to nanoseconds for filtering
start_time_ns = int(start_time.timestamp() * 1_000_000_000)
end_time_ns = int(end_time.timestamp() * 1_000_000_000)

# Filter traces data within the time window
filtered_traces = traces_data[
    (traces_data['startTime'] >= start_time_ns) & (traces_data['endTime'] <= end_time_ns)
]

# Identify traces related to the specific operation
operation_traces = filtered_traces[
    filtered_traces['spanName'] == 'inventory::/api/v1/inventory/batch-reconciliation'
]

# Analyze the call chain to find the most downstream faulty service
# Filter for spans with errors (statusCode != 0)
faulty_traces = filtered_traces[filtered_traces['statusCode'] != 0]

# Merge operation traces with faulty traces to trace the call chain
call_chain_analysis = operation_traces.merge(
    faulty_traces, left_on='spanId', right_on='parentSpanId', how='inner', suffixes=('_operation', '_faulty')
)

call_chain_analysis

```
Out[4]:
```
No traces related to the `inventory::/api/v1/inventory/batch-reconciliation` operation were found to have errors within the specified time window. Therefore, no downstream faulty service could be identified from the call chain analysis.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId_operation, spanId_operation, parentSpanId_operation, kind_operation, spanName_operation, startTime_operation, endTime_operation, duration_operation, serviceName_operation, pid_operation, ip_operation, hostname_operation, statusCode_operation, statusMessage_operation, traceState_operation, resources_operation, attributes_operation, __topic___operation, __tag__:__pack_id___operation, events_operation, links_operation, traceId_faulty, spanId_faulty, parentSpanId_faulty, kind_faulty, spanName_faulty, startTime_faulty, endTime_faulty, duration_faulty, serviceName_faulty, pid_faulty, ip_faulty, hostname_faulty, statusCode_faulty, statusMessage_faulty, traceState_faulty, resources_faulty, attributes_faulty, __topic___faulty, __tag__:__pack_id___faulty, events_faulty, links_faulty]
Index: []

[0 rows x 42 columns]```
```

In [ ]:
# In[5]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t094/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is in datetime format
logs_data['_time_'] = pd.to_datetime(logs_data['_time_'], errors='coerce')

# Filter logs within the time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= start_time) & (logs_data['_time_'] <= end_time)
]

# Search for logs containing error messages or keywords related to anomalies
keywords = ['error', 'frontend', 'frontend-proxy', 'node']
relevant_logs = filtered_logs[
    filtered_logs['content'].str.contains('|'.join(keywords), case=False, na=False)
]

relevant_logs

```
Out[5]:
```
No logs containing error messages or keywords related to anomalies (such as `frontend`, `frontend-proxy`, or Kubernetes nodes) were found within the specified time window.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```